In [0]:
SELECT CASE WHEN c.next_fk_content_id IS NOT NULL AND c.next_fk_content_id != 3468026 IS NOT NULL
             AND c.prev_fk_content_id IS NOT NULL AND c.prev_fk_content_id != 3468026 IS NOT NULL
             THEN CASE WHEN c.diff_from_prev <= 5 AND c.diff_from_next <= 5 THEN 'Between Detected Session'
                       WHEN c.diff_from_prev <= 5 THEN 'Between Detected Session - But next detected much later'
                       WHEN c.diff_from_next <= 5 THEN 'Between Detected Session - But prev detected much earlier'
                       ELSE 'Between Detected Session - But prev and next far from null' END
            WHEN c.next_fk_content_id IS NOT NULL AND c.next_fk_content_id != 3468026 IS NOT NULL
             AND c.prev_fk_content_id IS NULL
             THEN CASE WHEN c.diff_from_next <= 5 THEN 'Before First Detected Session'
                       ELSE 'Before First Detected Session - But next session far from null' END
            WHEN c.prev_fk_content_id IS NOT NULL AND c.prev_fk_content_id != 3468026 IS NOT NULL
             AND c.next_fk_content_id IS NULL
             THEN CASE WHEN c.diff_from_prev <= 5 THEN 'After Last Detected Session'
                       ELSE 'After Last Detected Session - But prev session far from null' END

FROM (
  SELECT vc.fk_tvid, vc.session_start, vc.session_end, vc.session_duration, vc.fk_content_id
  , LEAD(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end) AS next_start
  , LAG(session_end)    OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end) AS prev_end
  , TIMESTAMPDIFF(SECOND, prev_end, session_start) AS diff_from_prev
  , TIMESTAMPDIFF(SECOND, session_end, next_start) AS diff_from_next
  , LEAD(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end) AS next_fk_content_id
  , LAG(fk_content_id)  OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end) AS prev_fk_content_id
  FROM (
    SELECT vc.fk_tvid, vc.session_start, vc.session_end, vc.session_duration, vc.fk_content_id
    , LEAD(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end) AS next_start
    , LAG(session_end)    OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end) AS prev_end
    , LEAD(session_duration) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end) AS next_duration
    , LAG(session_duration)  OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end) AS prev_duration
    , CASE WHEN next_start < session_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', session_end) THEN 1
            WHEN prev_end > session_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', session_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
            ELSE CASE WHEN next_start IS NOT NULL AND next_start < session_end AND next_duration < session_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', session_start) THEN 1
                      WHEN prev_end IS NOT NULL AND prev_end > session_start AND prev_duration < session_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', session_end) THEN 1
        END END AS keep
    FROM prod.detection.viewing_content_firehose vc
    WHERE vc.session_start >= CURRENT_DATE - 1
      AND vc.session_start < CURRENT_DATE
  ) vc
  WHERE keep = 1
) c
WHERE fk_content_id = 3468026
  -- AND (next_fk_content_id IS NULL OR next_fk_content_id != 3468026)
  -- AND (prev_fk_content_id IS NULL OR prev_fk_content_id != 3468026)
  -- AND (diff_from_prev <= 5 OR prev_end IS NULL)
  -- AND (diff_from_next <= 5 OR next_start IS NULL)

In [0]:
SELECT session_type
, CASE WHEN comms.fk_tvid IS NOT NULL THEN 'Comm Playing' ELSE 'No Comms' END AS comm_or_not
-- , CASE WHEN diff_from_prev IS NULL THEN NULL
--        WHEN diff_from_prev <= 5 THEN '<=5'
--        WHEN diff_from_prev <= 10 THEN '6-10 seconds'
--        WHEN diff_from_prev <= 30 THEN '11-30 seconds'
--        WHEN diff_from_prev <= 60 THEN '31-60 seconds'
--        WHEN diff_from_prev <= 120 THEN '61-120 seconds'
--        WHEN diff_from_prev <= 300 THEN '2-5 Minutes'
--        WHEN diff_from_prev <= 600 THEN '5-10 Minutes'
--        WHEN diff_from_prev <= 1200 THEN '10-20 Minutes'
--        WHEN diff_from_prev <= 1800 THEN '20-30 Minutes'
--        WHEN diff_from_prev <= 3600 THEN '30-60 Minutes'
--        ELSE 'Over 1 Hour' END AS diff_from_prev_bucket
-- , CASE WHEN diff_to_next IS NULL THEN NULL
--        WHEN diff_to_next <= 5 THEN '<=5'
--        WHEN diff_to_next <= 10 THEN '6-10 seconds'
--        WHEN diff_to_next <= 30 THEN '11-30 seconds'
--        WHEN diff_to_next <= 60 THEN '31-60 seconds'
--        WHEN diff_to_next <= 120 THEN '61-120 seconds'
--        WHEN diff_to_next <= 300 THEN '2-5 Minutes'
--        WHEN diff_to_next <= 600 THEN '5-10 Minutes'
--        WHEN diff_to_next <= 1200 THEN '10-20 Minutes'
--        WHEN diff_to_next <= 1800 THEN '20-30 Minutes'
--        WHEN diff_to_next <= 3600 THEN '30-60 Minutes'
--        ELSE 'Over 1 Hour' END AS diff_to_next_bucket
, COUNT(DISTINCT tvid||'_'||ts_start) AS session_count
FROM (
SELECT tvid, ts_start, ts_end, ts_duration, cid, chan_callsign, epid
  , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
  , TIMESTAMPDIFF(SECOND, prev_end, ts_start) AS diff_from_prev
  , TIMESTAMPDIFF(SECOND, ts_end, next_start) AS diff_to_next
  , LEAD(cid) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_cid
  , LAG(cid) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_cid
  , CASE WHEN LOWER(next_cid) = 'unknown' THEN
              CASE WHEN LOWER(prev_cid) = 'unknown' THEN 'Between Null Sessions'
                   WHEN prev_cid IS NULL THEN 'First Session - Next Session Null'
                   WHEN LOWER(prev_cid) != 'unknown' THEN 'Prev Session Detected - Next Session Null' END
         WHEN LOWER(prev_cid) = 'unknown' THEN
              CASE WHEN next_cid IS NULL THEN 'Last Session - Prev Session Null'
                   WHEN LOWER(next_cid) != 'unknown' THEN 'Prev Session Null - Next Session Detected' END
         WHEN prev_cid IS NOT NULL AND LOWER(prev_cid) != 'unknown' THEN
              CASE WHEN LOWER(next_cid) = 'unknown' THEN 'Prev Session Detected - Next Session Null'
                   WHEN next_cid IS NULL THEN 'Last Session - Prev Session Detected'
                   WHEN LOWER(next_cid) != 'unknown' THEN 'Between Detected Sessions' END
         WHEN next_cid IS NOT NULL AND LOWER(next_cid) != 'unknown' THEN
              CASE WHEN LOWER(prev_cid) = 'unknown' THEN 'Prev Session Null - Next Session Detected'
                   WHEN prev_cid IS NULL THEN 'First Session - Next Session Detected'
                   WHEN LOWER(prev_cid) != 'unknown' THEN 'Between Detected Sessions' END
         WHEN prev_cid IS NULL THEN
              CASE WHEN next_cid IS NULL THEN 'Only Session'
                   WHEN LOWER(next_cid) = 'unknown' THEN 'First Session - Next Session Null'
                   WHEN LOWER(next_cid) != 'unknown' THEN 'First Session - Next Session Detected' END
  END AS session_type
FROM (
  SELECT *
  , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
  , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
  , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
  , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
         WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
    END AS overlapping_session
  , CASE WHEN overlapping_session IS NULL THEN 1
         ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                   WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
      END END AS keep
  FROM (
    SELECT tvid
    , ts_start
    , ts_end
    , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
    , cid
    , chan_callsign
    , epid
    , air_date
    , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
    FROM prod.staging.vizio_content_firehose
    WHERE ts_start >= CURRENT_DATE - 1
      AND ts_start < CURRENT_DATE

  ) vc
  WHERE rn = 1
) vc
WHERE keep = 1) vc
LEFT JOIN detection.viewing_commercials_firehose comms
  ON comms.fk_tvid = vc.tvid
 AND comms.session_start >= vc.ts_start
 AND comms.session_start <= vc.ts_end
 AND comms.session_start >= CURRENT_DATE - 1
  AND comms.session_start < CURRENT_DATE
WHERE LOWER(cid) = 'unknown'
  
GROUP BY 1, 2
ORDER BY 1, 2

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT session_type
, CASE WHEN comms.fk_tvid IS NOT NULL THEN 'Comm Playing' ELSE 'No Comms' END AS comm_or_not
-- , CASE WHEN diff_from_prev IS NULL THEN NULL
--        WHEN diff_from_prev <= 5 THEN '<=5'
--        WHEN diff_from_prev <= 10 THEN '6-10 seconds'
--        WHEN diff_from_prev <= 30 THEN '11-30 seconds'
--        WHEN diff_from_prev <= 60 THEN '31-60 seconds'
--        WHEN diff_from_prev <= 120 THEN '61-120 seconds'
--        WHEN diff_from_prev <= 300 THEN '2-5 Minutes'
--        WHEN diff_from_prev <= 600 THEN '5-10 Minutes'
--        WHEN diff_from_prev <= 1200 THEN '10-20 Minutes'
--        WHEN diff_from_prev <= 1800 THEN '20-30 Minutes'
--        WHEN diff_from_prev <= 3600 THEN '30-60 Minutes'
--        ELSE 'Over 1 Hour' END AS diff_from_prev_bucket
-- , CASE WHEN diff_to_next IS NULL THEN NULL
--        WHEN diff_to_next <= 5 THEN '<=5'
--        WHEN diff_to_next <= 10 THEN '6-10 seconds'
--        WHEN diff_to_next <= 30 THEN '11-30 seconds'
--        WHEN diff_to_next <= 60 THEN '31-60 seconds'
--        WHEN diff_to_next <= 120 THEN '61-120 seconds'
--        WHEN diff_to_next <= 300 THEN '2-5 Minutes'
--        WHEN diff_to_next <= 600 THEN '5-10 Minutes'
--        WHEN diff_to_next <= 1200 THEN '10-20 Minutes'
--        WHEN diff_to_next <= 1800 THEN '20-30 Minutes'
--        WHEN diff_to_next <= 3600 THEN '30-60 Minutes'
--        ELSE 'Over 1 Hour' END AS diff_to_next_bucket
, COUNT(DISTINCT tvid||'_'||ts_start) AS session_count
FROM (
SELECT tvid, ts_start, ts_end, ts_duration, cid, chan_callsign, epid
  , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
  , TIMESTAMPDIFF(SECOND, prev_end, ts_start) AS diff_from_prev
  , TIMESTAMPDIFF(SECOND, ts_end, next_start) AS diff_to_next
  , LEAD(cid) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_cid
  , LAG(cid) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_cid
  , CASE WHEN LOWER(next_cid) = 'unknown' THEN
              CASE WHEN LOWER(prev_cid) = 'unknown' THEN 'Between Null Sessions'
                   WHEN prev_cid IS NULL THEN 'First Session - Next Session Null'
                   WHEN LOWER(prev_cid) != 'unknown' THEN 'Prev Session Detected - Next Session Null' END
         WHEN LOWER(prev_cid) = 'unknown' THEN
              CASE WHEN next_cid IS NULL THEN 'Last Session - Prev Session Null'
                   WHEN LOWER(next_cid) != 'unknown' THEN 'Prev Session Null - Next Session Detected' END
         WHEN prev_cid IS NOT NULL AND LOWER(prev_cid) != 'unknown' THEN
              CASE WHEN LOWER(next_cid) = 'unknown' THEN 'Prev Session Detected - Next Session Null'
                   WHEN next_cid IS NULL THEN 'Last Session - Prev Session Detected'
                   WHEN LOWER(next_cid) != 'unknown' THEN 'Between Detected Sessions' END
         WHEN next_cid IS NOT NULL AND LOWER(next_cid) != 'unknown' THEN
              CASE WHEN LOWER(prev_cid) = 'unknown' THEN 'Prev Session Null - Next Session Detected'
                   WHEN prev_cid IS NULL THEN 'First Session - Next Session Detected'
                   WHEN LOWER(prev_cid) != 'unknown' THEN 'Between Detected Sessions' END
         WHEN prev_cid IS NULL THEN
              CASE WHEN next_cid IS NULL THEN 'Only Session'
                   WHEN LOWER(next_cid) = 'unknown' THEN 'First Session - Next Session Null'
                   WHEN LOWER(next_cid) != 'unknown' THEN 'First Session - Next Session Detected' END
  END AS session_type
FROM (
  SELECT *
  , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
  , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
  , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
  , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
         WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
    END AS overlapping_session
  , CASE WHEN overlapping_session IS NULL THEN 1
         ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                   WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
      END END AS keep
  FROM (
    SELECT tvid
    , ts_start
    , ts_end
    , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
    , cid
    , chan_callsign
    , epid
    , air_date
    , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
    FROM prod.cooker.vizio_content_firehose
    WHERE ts_start >= CURRENT_DATE - 1
      AND ts_start < CURRENT_DATE

  ) vc
  WHERE rn = 1
) vc
WHERE keep = 1) vc
LEFT JOIN detection.viewing_commercials_firehose comms
  ON comms.fk_tvid = vc.tvid
 AND comms.session_start >= vc.ts_start
 AND comms.session_start <= vc.ts_end
 AND comms.session_start >= CURRENT_DATE - 1
  AND comms.session_start < CURRENT_DATE
WHERE LOWER(cid) = 'unknown'
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
SELECT session_type
, CASE WHEN diff_from_prev IS NULL THEN NULL
       WHEN diff_from_prev <= 5 THEN '<=5'
       WHEN diff_from_prev <= 10 THEN '6-10 seconds'
       WHEN diff_from_prev <= 30 THEN '11-30 seconds'
       WHEN diff_from_prev <= 60 THEN '31-60 seconds'
       WHEN diff_from_prev <= 120 THEN '61-120 seconds'
       WHEN diff_from_prev <= 300 THEN '2-5 Minutes'
       WHEN diff_from_prev <= 600 THEN '5-10 Minutes'
       WHEN diff_from_prev <= 1200 THEN '10-20 Minutes'
       WHEN diff_from_prev <= 1800 THEN '20-30 Minutes'
       WHEN diff_from_prev <= 3600 THEN '30-60 Minutes'
       ELSE 'Over 1 Hour' END AS diff_from_prev_bucket
, CASE WHEN diff_to_next IS NULL THEN NULL
       WHEN diff_to_next <= 5 THEN '<=5'
       WHEN diff_to_next <= 10 THEN '6-10 seconds'
       WHEN diff_to_next <= 30 THEN '11-30 seconds'
       WHEN diff_to_next <= 60 THEN '31-60 seconds'
       WHEN diff_to_next <= 120 THEN '61-120 seconds'
       WHEN diff_to_next <= 300 THEN '2-5 Minutes'
       WHEN diff_to_next <= 600 THEN '5-10 Minutes'
       WHEN diff_to_next <= 1200 THEN '10-20 Minutes'
       WHEN diff_to_next <= 1800 THEN '20-30 Minutes'
       WHEN diff_to_next <= 3600 THEN '30-60 Minutes'
       ELSE 'Over 1 Hour' END AS diff_to_next_bucket
, COUNT(*)
FROM (
SELECT tvid, ts_start, ts_end, ts_duration, cid, chan_callsign, epid
  , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
  , TIMESTAMPDIFF(SECOND, prev_end, ts_start) AS diff_from_prev
  , TIMESTAMPDIFF(SECOND, ts_end, next_start) AS diff_to_next
  , LEAD(cid) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_cid
  , LAG(cid) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_cid
  , CASE WHEN LOWER(next_cid) = 'unknown' THEN
              CASE WHEN LOWER(prev_cid) = 'unknown' THEN 'Between Null Sessions'
                   WHEN prev_cid IS NULL THEN 'First Session - Next Session Null'
                   WHEN LOWER(prev_cid) != 'unknown' THEN 'Prev Session Detected - Next Session Null' END
         WHEN LOWER(prev_cid) = 'unknown' THEN
              CASE WHEN next_cid IS NULL THEN 'Last Session - Prev Session Null'
                   WHEN LOWER(next_cid) != 'unknown' THEN 'Prev Session Null - Next Session Detected' END
         WHEN prev_cid IS NOT NULL AND LOWER(prev_cid) != 'unknown' THEN
              CASE WHEN LOWER(next_cid) = 'unknown' THEN 'Prev Session Detected - Next Session Null'
                   WHEN next_cid IS NULL THEN 'Last Session - Prev Session Detected'
                   WHEN LOWER(next_cid) != 'unknown' THEN 'Between Detected Sessions' END
         WHEN next_cid IS NOT NULL AND LOWER(next_cid) != 'unknown' THEN
              CASE WHEN LOWER(prev_cid) = 'unknown' THEN 'Prev Session Null - Next Session Detected'
                   WHEN prev_cid IS NULL THEN 'First Session - Next Session Detected'
                   WHEN LOWER(prev_cid) != 'unknown' THEN 'Between Detected Sessions' END
         ELSE 'IDK' END AS session_type
FROM (
  SELECT *
  , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
  , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
  , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
  , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
         WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
    END AS overlapping_session
  , CASE WHEN overlapping_session IS NULL THEN 1
         ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                   WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
      END END AS keep
  FROM (
    SELECT tvid
    , ts_start
    , ts_end
    , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
    , cid
    , chan_callsign
    , epid
    , air_date
    , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
    FROM prod.staging.vizio_content_firehose
    WHERE ts_start >= CURRENT_DATE - 1
      AND ts_start < CURRENT_DATE

  ) vc
  WHERE rn = 1
) vc
WHERE keep = 1) vc
WHERE LOWER(cid) = 'unknown'
GROUP BY 1, 2, 3
ORDER BY 3, 1, 2

In [0]:
SELECT 'DP4', fk_content_id IS NULL AS null_content_id, fk_content_id = 3468026 AS null_session, fk_content_id != 3468026 AS detected_session
, SUM(session_duration)/3600.0 AS ttl_duration
FROM prod.detection.viewing_content_firehose
WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 27 HOURS
  AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  AND fk_zoo_id = 17
GROUP BY 1, 2, 3, 4
UNION
SELECT 'DP5', fk_content_id IS NULL AS null_content_id, fk_content_id = 3468026 AS null_session, fk_content_id != 3468026 AS detected_session
, SUM(session_duration)/3600.0 AS ttl_duration
FROM qa.detection.viewing_content_firehose
WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 27 HOURS
  AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  AND fk_zoo_id = 17
GROUP BY 1, 2, 3, 4

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT dma_name
, fk_content_id IS NULL AS null_content_id
, SUM(session_duration)/3600.0 AS ttl_duration
FROM qa.detection.viewing_content_firehose vc
JOIN detection.dma
  ON dma.dma_id = vc.fk_dma_id
WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 27 HOURS
  AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  AND fk_zoo_id = 17
GROUP BY 1, 2

In [0]:
SELECT landing.cid, COUNT(*)
FROM prod.cooker.vizio_content_firehose AS landing
JOIN qa.detection.viewing_content_firehose AS vcf
  ON vcf.fk_tvid = landing.tvid
 AND vcf.session_start = landing.ts_start
 AND vcf.session_end = landing.ts_end
WHERE landing.ts_start >= CURRENT_DATE - 1
  AND landing.ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 6 HOURS
  AND vcf.session_start >= CURRENT_DATE - 1
  AND vcf.session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 6 HOURS
  AND vcf.fk_content_id IS NULL
GROUP BY 1

In [0]:
SELECT landing.cid, landing.epid, landing.chan_callsign, landing.air_date, COUNT(*)
FROM prod.staging.vizio_content_firehose AS landing
WHERE landing.ts_start >= CURRENT_DATE - 1
  AND landing.ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 6 HOURS
  AND cid = '16333711452_FBNP_2025-06-09T13:00:00Z'
GROUP BY 1, 2, 3, 4

In [0]:
SELECT landing.cid, landing.epid, landing.chan_callsign, landing.air_date, COUNT(*)
FROM prod.cooker.vizio_content_firehose AS landing
JOIN qa.detection.viewing_content_firehose AS vcf
  ON vcf.fk_tvid = landing.tvid
 AND vcf.session_start = landing.ts_start
 AND vcf.session_end = landing.ts_end
WHERE landing.ts_start >= CURRENT_DATE - 1
  AND landing.ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 6 HOURS
  AND vcf.session_start >= CURRENT_DATE - 1
  AND vcf.session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 6 HOURS
  AND vcf.fk_content_id IS NULL
  AND cid = '16333711452_FBNP_2025-06-09T13:00:00Z'
GROUP BY 1, 2, 3, 4

In [0]:
SELECT * FROM prod.detection.content_ids_firehose WHERE content_cid = '16333711452_FBNP_2025-06-09T13:00:00Z'

In [0]:
SELECT * FROM prod.detection.content_ids_firehose WHERE content_cid LIKE '16333711452_%'

In [0]:
SELECT * FROM detection.epg_show WHERE database_key = '16333711452'

In [0]:
SELECT * FROM detection.inscape_station_map WHERE inscape_call_sign = 'FBNP'

In [0]:
SELECT * FROM detection.epg_schedule
WHERE fk_show_id = 15144794
  AND fk_station_id = 95284
  AND airdate = '2025-06-09T13:00:00'

In [0]:
SELECT * FROM detection.inscape_station_map
WHERE mapped_vendor_station_id IN (154598, 
147494, 
133218, 
130689, 
95284, 
92576)

In [0]:
SELECT * FROM prod.detection.content_ids_firehose WHERE content_cid LIKE '%FBNP_2025-06-09%'
LIMIT 100

In [0]:
SELECT dp4.fk_tvid, dp4.session_start, dp4.session_end, dp4.session_duration
, dp4.airdate, dp4.media_time_start, dp4.media_time_end
, dp4.fk_show_id, dp4.fk_station_id, dp4.runtime, dp4.fk_content_id
FROM qa.detection.viewing_content_firehose dp4
WHERE fk_content_id IS NULL
  AND session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 27 HOURS
  AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  AND fk_zoo_id = 17
  AND is_live = False
  AND airdate <= session_start - INTERVAL 24 HOUR
ORDER BY 1, 2
LIMIT 1000

In [0]:
WITH missing_content_id_from_dp5 AS (
  SELECT fk_tvid, session_start, session_end
  FROM qa.detection.viewing_content_firehose
  WHERE fk_content_id IS NULL
    AND session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 24 HOURS
    AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 1 HOURS
    AND fk_zoo_id = 17
)
SELECT dp4.fk_tvid, dp4.session_start, dp4.session_end, dp4.session_duration
, dp4.media_time_start, dp4.media_time_end
, dp4.fk_show_id, dp4.fk_station_id, dp4.runtime, dp4.fk_content_id
, dp4.vizio_epg_airing, dp4.vizio_epg_station, dp4.vizio_epg_program
, dp4.tuner_channel_id, dp4.tuner_schedule_id, dp4.tuner_program_id
FROM missing_content_id_from_dp5 mci
JOIN prod.detection.viewing_content_firehose dp4
  ON mci.fk_tvid = dp4.fk_tvid
 AND mci.session_start = dp4.session_start
 AND mci.session_end = dp4.session_end
WHERE dp4.session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 24 HOURS
  AND dp4.session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 1 HOURS
  AND dp4.fk_zoo_id = 17
ORDER BY dp4.fk_tvid, dp4.session_start
LIMIT 1000